# IMU Grid-Search Results Analysis

Inspect the pickled outputs of `imu/scripts/grid_search.py` (`grid_search_results_*.pkl`). Surface F1 scores, best hyperparameters, and Random Forest feature importances per split (balanced / LOSO / signant); aggregate and persist a summary (`feature_importance_data.pkl`) for downstream plotting.

## Configuration

In [ ]:
# Set to the dataset family whose grid-search pickle you want to load.
BEST_DATASET = "temporal"
RESULTS_GLOB = "grid_search_results_*.pkl"
FEATURE_NAMES_GLOB = "feature_names*.txt"
CHRONOS_RESULT = "grid_search_results_temporal_chronos.pkl"
IMPORTANCE_OUTPUT = "feature_importance_data.pkl"


## Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
from tabulate import tabulate
import glob
import matplotlib.pyplot as plt
import matplotlib
print(matplotlib.style.available)

## Load feature-name dictionary

In [ ]:
features_names_dict = {}

for features_file in glob.glob(FEATURE_NAMES_GLOB):
    with open(features_file, "r") as f:
        feature_names = f.readlines()
        features_names_dict[features_file.split(".")[0]] = list(filter(lambda x: (not x in ["subject", "label","Unnamed: 0"]),[name.strip() for name in feature_names]))

## Inspect a single results pickle

In [ ]:
database = CHRONOS_RESULT

handle = pickle.load(open(database, 'rb'))

#balanced_results = handle["balanced_part"]
loso_results = handle["loso_part"]
#signant_results = handle["signant_part"]

"""balanced_agg = {}
for k,v in balanced_results.items():
    f1_score = v["f1_score"]
    model_name,database = k.split("/")
    model_name = model_name.split("-")[0]
    database = database.split(".")[0]
    balanced_agg[model_name] = f1_score
"""
print("RANDOM PARTITION\n")

#print(tabulate(balanced_agg.items(),["Model","F1 Score"]))
print("--"*30)

loso_results_agg = {}

for k,v in loso_results.items():
    f1_score = v["f1_score"]
    model_name,subject = k.split("-")
    loso_results_agg[model_name] = loso_results_agg.get(model_name, [])
    loso_results_agg[model_name].append(f1_score)

for model_name, scores in loso_results_agg.items():
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    loso_results_agg[model_name] = f"{mean_score:.2f} +- {std_score:.2f}"

print("LOSO RESULTS")
print(tabulate(loso_results_agg.items(),["Model","F1 Score"]))
print("--"*30)
"""signant_agg = {}
for k,v in signant_results.items():
    f1_score = v["f1_score"]
    model_name,db = k.split("-")
    signant_agg[model_name] = f1_score

print("SIGNANT-NONSIGNANT")
print(tabulate(signant_agg.items(),["Model","F1 Score"]))
print("--"*30)"""

## Load the best dataset's results

In [ ]:
best = BEST_DATASET
best_results = pickle.load(open(f"grid_search_results_{best}.pkl", 'rb'))

## Aggregate Random Forest feature importances

In [ ]:
feature_importances_dict = {}
for key in best_results.keys():
    feature_importances_dict[key] = {}
    for model_key in best_results[key].keys():
        model_info = best_results[key][model_key]
        if "feature_importances" in model_info:
            feature_importances = model_info["feature_importances"]
            feature_importances_dict[key][model_key] = dict(zip(features_names_dict[f"feature_names_{best}"],feature_importances))

In [ ]:
agg_res_importance = {}

for key, value in feature_importances_dict["balanced_part"][f"rf-FEATURES_NEW/{best}.csv"].items():
    movement = key.split("_")
    movement_name = "_".join([movement[0], movement[1], movement[2]])
    agg_res_importance["balanced_part"] = agg_res_importance.get("balanced_part", {})
    agg_res_importance["balanced_part"][movement_name] = agg_res_importance["balanced_part"].get(movement_name, 0) + value

for key, value in feature_importances_dict["loso_part"].items():
    for sub_key, sub_value in value.items():
        movement = sub_key.split("_")
        movement_name = "_".join([movement[0], movement[1], movement[2]])
        agg_res_importance["loso_part"] = agg_res_importance.get("loso_part", {})
        agg_res_importance["loso_part"][movement_name] = agg_res_importance["loso_part"].get(movement_name, [])
        agg_res_importance["loso_part"][movement_name].append(sub_value)

for key, value in feature_importances_dict["signant_part"]["rf-deaf"].items():
    movement = key.split("_")
    movement_name = "_".join([movement[0], movement[1], movement[2]])
    agg_res_importance["signant_part"] = agg_res_importance.get("signant_part", {})
    agg_res_importance["signant_part"][movement_name] = agg_res_importance["signant_part"].get(movement_name, 0) + value

for key in agg_res_importance["loso_part"]:
    agg_res_importance["loso_part"][key] = np.sum(agg_res_importance["loso_part"][key])/5

for key in agg_res_importance.keys():
    agg_res_importance[key] = dict(sorted(agg_res_importance[key].items(), key=lambda item: item[1], reverse=True))



## Format for plotting

In [ ]:
limit = 4
matplotlib.style.use('seaborn-v0_8-darkgrid')

data = {}

for key in agg_res_importance.keys():
    data[key] = {}
    name = key.split("_")[0]
    if name.lower() == "balanced":
        name = "80-20 split"
    x_names = list(agg_res_importance[key].keys())[:limit]
    x_names = list(map(lambda x: " ".join(list(map(lambda x: x.capitalize(), x.split("_")))), x_names))
    x_vals = list(agg_res_importance[key].values())[:limit]
    data[key]["x_names"] = x_names
    data[key]["x_vals"] = x_vals
    plt.figure(figsize=(8,5))
    plt.bar(x_names, x_vals)
    plt.title(f"Feature Importances - {name.upper()}", fontsize=20)
    plt.xlabel("Features", fontsize=16)
    plt.ylabel("Importance", fontsize=16)
    plt.xticks(rotation=45, fontsize=14)
    plt.yticks(fontsize=14)
    plt.setp(plt.gca().get_xticklabels(), ha='right')
    plt.tight_layout()
    plt.show()

## Save summary

In [ ]:
with open(IMPORTANCE_OUTPUT, "wb") as f:
    pickle.dump(data, f)